In [1]:
!git clone https://github.com/nnam128/Scholarly-Label-Predictor.git

%cd Scholarly-Label-Predictor

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.utils.utils import load_csv

Cloning into 'Scholarly-Label-Predictor'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 152 (delta 48), reused 144 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 20.08 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/Scholarly-Label-Predictor


In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OMP_NUM_THREADS'] = '1'
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.utils.utils import load_csv
from src.models.train_bert import BERTTrainer

from src.config import (
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH,
    TRAIN_LABEL_PATH,
    SUBMISSION_BERT_PATH,
)

Using device: cuda


In [3]:
# Load data
train_df = load_csv(CLEANED_TRAIN_PATH)
test_df  = load_csv(CLEANED_TEST_PATH)

# Gắn label vào train_df
labels_df = load_csv(TRAIN_LABEL_PATH)
train_df['label'] = labels_df.values.ravel()

print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
print(f'Columns     : {train_df.columns.tolist()}')
print(f'\nLabel distribution (1-5):')
print(train_df['label'].value_counts().sort_index())

Train shape : (2496, 9)
Test shape  : (596, 7)
Columns     : ['id', 'title', 'venue', 'year', 'authors', 'doi', 'Label', 'abstract', 'label']

Label distribution (1-5):
label
1    904
2    514
3    439
4    367
5    272
Name: count, dtype: int64


In [ ]:
print('\n====================')
print('MODEL: SciBERT + Ordinal Classification')
print('====================')

# Khởi tạo Trainer
trainer = BERTTrainer(
    model_name="allenai/specter2",
    num_classes=5,
    early_stopping = 3,
    max_length=256,
    batch_size=16,
    lr=2e-5,
    n_splits=5,
)


MODEL: SciBERT + Ordinal Classification


In [ ]:
# Huấn luyện 5-fold Stratified CV
macro_f1_cv = trainer.train(train_df, label_col='label')

print(f'CV Macro F1-Score: {macro_f1_cv:.4f}')


Fold 1/5
  Epoch  1/17  loss=1.5622  val_macro_f1=0.1063  ✔ best
  Epoch  2/17  loss=1.4432  val_macro_f1=0.3315  ✔ best
  Epoch  3/17  loss=1.2771  val_macro_f1=0.3623  ✔ best
  Epoch  4/17  loss=1.1051  val_macro_f1=0.4090  ✔ best
  Epoch  5/17  loss=0.8920  val_macro_f1=0.3932    (no improve 1/2)
  Epoch  6/17  loss=0.6657  val_macro_f1=0.4085    (no improve 2/2)
  ⏹ Early stopping tại epoch 6
  → Best val Macro F1: 0.4090

Fold 2/5
  Epoch  1/17  loss=1.5477  val_macro_f1=0.1065  ✔ best
  Epoch  2/17  loss=1.4669  val_macro_f1=0.2567  ✔ best
  Epoch  3/17  loss=1.3209  val_macro_f1=0.2643  ✔ best
  Epoch  4/17  loss=1.1430  val_macro_f1=0.4109  ✔ best
  Epoch  5/17  loss=0.9198  val_macro_f1=0.4098    (no improve 1/2)
  Epoch  6/17  loss=0.6746  val_macro_f1=0.4124  ✔ best
  Epoch  7/17  loss=0.4920  val_macro_f1=0.4217  ✔ best
  Epoch  8/17  loss=0.3550  val_macro_f1=0.4067    (no improve 1/2)
  Epoch  9/17  loss=0.2156  val_macro_f1=0.4044    (no improve 2/2)
  ⏹ Early stopping 

In [ ]:
# Lưu checkpoints để dùng lại, không cần train lại
trainer.save_models(str(PROJECT_ROOT / 'models' / 'saved' / 'bert_folds'))
print('✔ Saved all fold checkpoints')

Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_1.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_2.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_3.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_4.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_5.pt
✔ Saved all fold checkpoints


In [ ]:
# Dự đoán trên tập Test
# Ensemble softmax avg từ tất cả fold models → nhãn 1-5
y_test_pred = trainer.predict(test_df)

print(f'Predictions shape: {y_test_pred.shape}')
print(f'Predicted classes : {np.unique(y_test_pred)}')

Predictions shape: (596,)
Predicted classes : [1 2 3 4 5]


In [ ]:
# Tạo và lưu submission
submission = pd.DataFrame({
    'id'   : test_df['id'],
    'Label': y_test_pred,
})

os.makedirs(Path(SUBMISSION_BERT_PATH).parent, exist_ok=True)
submission.to_csv(SUBMISSION_BERT_PATH, index=False)
print(f'✔ Saved BERT submission → {SUBMISSION_BERT_PATH}')

print('\nPredicted labels distribution (Should be 1-5):')
print(submission['Label'].value_counts().sort_index())

✔ Saved BERT submission → /Users/nhatnam/Documents/DM_252/Assignment/data/submission/submission_bert.csv

Predicted labels distribution (Should be 1-5):
Label
1    221
2    156
3     97
4     81
5     41
Name: count, dtype: int64
